# 02 — Preprocessing

**CSE437: Data Science | Group 15**

**Step 5 complete: eligibility and direct leakage removal. Responsible: Faraaz. Next: Step 6, evaluation split.**

This notebook currently implements only the deterministic eligibility stage. Step 6 split construction and Step 7 fitted preprocessing are pending. The original project brief and research questions remain unchanged.

Eligible does not mean fully processed or verified for use at booking time. No model has been trained, and no test set has been selected. This implementation was prepared with ChatGPT/Codex assistance; the assigned member must review and explain it.


**Execution:** All code cells below ran sequentially in a fresh Python process using IPython with actual outputs saved. A separate fresh Jupyter-kernel run is still required for final submission verification.


## 1. Decisions before execution

| Item | Step 5 decision | Reason / later action |
| --- | --- | --- |
| `reservation_status`, `reservation_status_date` | Exclude from candidate predictors | They describe the outcome; never use either column or derivatives in a model. |
| `is_canceled` | Export separately as target | The target must never be a predictor. |
| Known zero total guests | Exclude when all three counts are known and their sum equals zero | Restricts the analytic cohort to bookings with some known or potentially present guests. This is a scope choice, not proof of a recording error. |
| Missing child counts | Retain as unknown | A missing value must not be interpreted as zero guests. |
| Zero adults with positive total guests | Retain and flag | Zero adults alone does not mean an empty booking. |
| Identical records | Retain and assign candidate-predictor groups | Without a booking identifier, accidental copies cannot be distinguished from repeated legitimate reservations. |
| Negative ADR | Retain booking and flag | Step 7 will treat negative ADR as unavailable under a nonnegative-price assumption, then impute within training folds. Raw/current candidate value remains unchanged. |
| Zero ADR / zero nights | Retain and flag | These may be legitimate or cancellation-related; deleting them could bias the task. |
| High positive ADR | Retain; values above 1,000 get a review-only flag | 1,000 is descriptive, not a cleaning threshold. Any learned clipping/scaling must be selected on development folds. |

All rules are independent of the cancellation label and reservation statuses. Raw records remain untouched. Counts are cohort auditing, not model evaluation.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/eligibility.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository or its notebooks folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.eligibility import run_step5, SOURCE_SHA256, LEAKAGE_COLUMNS, TARGET

RAW_PATH = ROOT / 'data/raw/hotel_bookings.csv'
if not RAW_PATH.is_file():
    raise FileNotFoundError('Place the original CSV in data/raw/. See data/README.md.')
OUTPUT_DIR = ROOT / 'data/processed'
source_before = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
assert source_before == SOURCE_SHA256


## 2. Generate the eligible cohort and separate files

The implementation is in `src/eligibility.py`; it can also run using `python -m src.eligibility` from the repository root. Gzip compression preserves CSV contents, and `pandas.read_csv` reads it directly. No learned transformation is fitted here.


In [2]:
X, y, metadata, excluded, summary = run_step5(RAW_PATH, OUTPUT_DIR)
display(pd.DataFrame([
    {'stage': 'Original source', 'rows': summary['input_rows'], 'candidate_predictors': 29},
    {'stage': 'Excluded: known zero guests', 'rows': len(excluded), 'candidate_predictors': None},
    {'stage': 'Eligible cohort', 'rows': len(X), 'candidate_predictors': X.shape[1]},
]))
display(pd.DataFrame({'removed_from_predictors': [TARGET, *LEAKAGE_COLUMNS],
                      'purpose': ['Separate target', 'Outcome leakage', 'Outcome leakage']}))


,stage,rows,candidate_predictors
0,Original source,119390,29.0
1,Excluded: known zero guests,180,NaN
2,Eligible cohort,119210,29.0


,removed_from_predictors,purpose
0,is_canceled,Separate target
1,reservation_status,Outcome leakage
2,reservation_status_date,Outcome leakage


## 3. Retained anomalies and repeated records

The flags below live only in metadata. They are not added to the predictor matrix. Candidate grouping excludes both statuses and the target; this prevents group identity from being derived from the answer. Distinct bookings may still share predictors and have different outcomes: retain both labels and keep the group together.


In [3]:
display(pd.Series(summary['retained_flags'], name='Retained rows').to_frame())
display(pd.Series({key: summary[key] for key in [
    'original_exact_duplicate_copies_retained', 'candidate_duplicate_groups',
    'candidate_duplicate_extra_copies', 'candidate_rows_in_repeated_groups',
    'candidate_groups_with_conflicting_labels']}, name='Count').to_frame())
display(pd.Series(summary['retained_target_counts'], name='Cohort label counts').to_frame())


,Retained rows
unknown_guest_total,4
zero_adults_positive_guests,223
zero_total_nights,645
negative_adr,1
zero_adr,1810
adr_above_1000_review_only,1


,Count
original_exact_duplicate_copies_retained,31980
candidate_duplicate_groups,86707
candidate_duplicate_extra_copies,32503
candidate_rows_in_repeated_groups,40730
candidate_groups_with_conflicting_labels,265


,Cohort label counts
0,75011
1,44199


## 4. Verify row alignment, exclusions, and original values

All three retained CSV files use the same row order. `source_row_id` is the zero-based record position after the raw header, not a real booking identifier. Use it for tracing and split assignment, never as a predictor. Only the known-zero-guest rule removes records.


In [4]:
raw = pd.read_csv(RAW_PATH)
expected = raw.iloc[metadata['source_row_id']].reset_index(drop=True)
pd.testing.assert_frame_equal(X, expected.drop(columns=[TARGET, *LEAKAGE_COLUMNS]))
pd.testing.assert_frame_equal(y, expected[[TARGET]])
assert len(X) == len(y) == len(metadata) == 119210
assert X.shape[1] == 29
assert len(excluded) == 180
assert not set(metadata['source_row_id']).intersection(excluded['source_row_id'])
assert sorted(metadata['source_row_id'].tolist() + excluded['source_row_id'].tolist()) == list(range(len(raw)))
assert not ({TARGET, *LEAKAGE_COLUMNS} & set(X.columns))
assert metadata.groupby('duplicate_group_id')['arrival_date'].nunique().max() == 1
assert summary['retained_flags']['unknown_guest_total'] == 4
assert summary['retained_flags']['zero_adults_positive_guests'] == 223
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == source_before
print('PASS: complete row accounting; candidates/labels aligned; source values unchanged; direct leakage fields absent.')
print('PASS: every duplicate group belongs to one arrival date. Step 6 must enforce nonoverlapping groups.')


PASS: complete row accounting; candidates/labels aligned; source values unchanged; direct leakage fields absent.
PASS: every duplicate group belongs to one arrival date. Step 6 must enforce nonoverlapping groups.


## 5. Check written artifacts

The committed summary records source/output hashes, dimensions, and exact runtime versions. Reading the compressed CSVs needs no additional dependency. Boolean flags remain separate from model inputs.


In [5]:
for filename, record in summary['outputs'].items():
    path = OUTPUT_DIR / filename
    assert hashlib.sha256(path.read_bytes()).hexdigest() == record['sha256']
    loaded = pd.read_csv(path)
    assert loaded.shape == (record['rows'], record['columns'])
display(pd.DataFrame.from_dict(summary['outputs'], orient='index')[['rows', 'columns', 'bytes']])
print('PASS: every saved output reloads with the documented dimensions and checksum.')


PASS: every saved output reloads with the documented dimensions and checksum.


,rows,columns,bytes
step5_candidates.csv.gz,119210,29,1001550
step5_target.csv.gz,119210,1,5087
step5_metadata.csv.gz,119210,10,625643
step5_exclusions.csv,180,2,5495


## 6. Handoff and subsequent preprocessing commitments

**Step 6 — Faraaz:** select a whole-arrival-date chronological holdout; freeze all assignments before predictor-outcome EDA; use forward development validation; assert every retained duplicate group remains in a single partition for each split. The same-date group check makes whole-date boundaries feasible, but no boundary has been selected in this step.

**Step 7 — Faraaz:** handle missingness, encode categories, and fit imputers/scalers only within training folds. Planned agency/company handling: `agent` is categorical with an explicit no-agent category under the source NULL convention; discard the sparse `company` identifier from the primary model, with a company-presence indicator considered during Step 9. These are commitments, not transformations already applied. Keep country unknowns distinct from agency/company non-applicability. Handle negative ADR as unavailable under the documented price assumption; keep zero and high positive ADR. Report sensitivity if adopting a train-fitted high-price transform/cap.

**Prediction-time review:** the source does not supply reliable snapshots of every field at booking creation. Review `assigned_room_type`, `booking_changes`, `days_in_waiting_list`, and other potentially updated values before finalizing model inputs. Removing the two direct leakage fields alone does not establish deployment-time availability. The 29 columns are candidates, not the final feature set.

**Analysis focus, without rewriting the questions:** before development EDA/modeling, expect longer lead time and prior cancellations to be associated with greater cancellation risk, and investigate differences by deposit type. Treat these as hypotheses and associations, not proven causal effects.

**Limitations:** zero-guest exclusion changes the population; retained repeats give more weight to frequently occurring records; sparse/high-price/unknown records remain; full-source quality auditing has already been seen. Report these choices honestly. A later split will not undo the fact that basic full-source audit summaries have been inspected.

Sources: [approved dataset](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand), [original documentation](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/), [Antonio et al. (2019)](https://doi.org/10.1016/j.dib.2018.11.126).

OpenAI ChatGPT/Codex assisted with policy documentation, code, execution, and verification. Contributions in the final report must reflect actual member work.
